[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_03_backprop_and_optimizers/task_2_backprop.ipynb)

# Week 3 · Task 2: Backpropagation from scratch

Last week you built the **forward pass** of a multi-layer perceptron with plain tensors. This week the network
learns: you implement the **backward pass** for every building block of our little framework (linear layer,
activations, loss functions, the model container) and verify each piece against PyTorch autograd and against a
numerical gradient check.

## What you will do
1. **Linear layer** - implement `backward`: the parameter gradients `dW`, `db` and the gradient handed to the previous layer.
2. **Activations** - implement `backward` for Sigmoid, Tanh, ReLU and LeakyReLU.
3. **Loss functions** - implement the squared error and the binary cross-entropy, forward and backward.
4. **Model** - chain the `backward` calls of all layers in reverse order.
5. **Gradient check** - run a full forward/backward pass on the Circles dataset and confirm the gradients numerically.
6. **Questions** - check your understanding.

## How to work through this notebook
- Every place that needs your input is marked with a `# TODO` comment and/or `...`. **Replace every `...` with your own code.**
- Written answers go into the markdown cells marked _Your answer here._ (double-click the cell to edit it).
- After every implementation there is a **verification cell** that compares your code with PyTorch autograd and
  ends with an `assert`. `OK` means every element of your tensor agrees with the reference to within `1e-10`
  (absolute tolerance, no relative tolerance; `float64` makes this a tight but fair bar). You are done with a part when every line prints `OK`; a `MISMATCH` stops the cell with an
  `AssertionError`, so a notebook that runs top to bottom without errors has a backward pass that agrees with autograd.
- Run the cells **in order**: later cells depend on classes defined earlier.
- Our framework uses `torch.float64` tensors and **no autograd**. Autograd appears only in the verification cells,
  where it plays the role of the reference answer.

Working in Colab? Replace `fiit-ba` in the URL with your GitHub username to open the copy in your fork, and run the
setup cell below.

_Credit: this notebook is a PyTorch port of the NSIETE (FIIT STU) numpy lab "Task 2 - backprop"._

In [ ]:
# Colab setup (no-op locally)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

In [ ]:
%matplotlib inline
import math
from collections import OrderedDict

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

In [ ]:
# --- Configuration ---
SEED = 42
DTYPE = torch.float64   # double precision keeps the numerical gradient check tight (see question 9.1 at the end)

torch.manual_seed(SEED)

In [ ]:
def check(name: str, mine, ref: torch.Tensor, atol: float = 1e-10) -> bool:
    """Compare your tensor with a reference tensor and print one OK / MISMATCH line.

    OK means every element differs by at most `atol` (absolute tolerance, rtol=0).
    """
    if not torch.is_tensor(mine):
        print(f"MISMATCH  {name:<22s} expected a tensor, got {type(mine).__name__}")
        return False
    if tuple(mine.shape) != tuple(ref.shape):
        print(f"MISMATCH  {name:<22s} shape {tuple(mine.shape)} != expected {tuple(ref.shape)}")
        return False
    mine = mine.to(ref.dtype)
    diff = (mine - ref).abs().max().item()
    ok = torch.allclose(mine, ref, atol=atol, rtol=0.0)
    print(f"{'OK        ' if ok else 'MISMATCH  '}{name:<22s} max |diff| = {diff:.1e}")
    return ok

## 1. Recap: forward and backward through one layer

**Notation (column convention, as in the lecture).** A batch of $m$ samples is stored **column-wise**: the input of
layer $l$ is $A^{[l-1]}$ of shape $(n_{l-1}, m)$, the weights are $W^{[l]}$ of shape $(n_l, n_{l-1})$ and the bias
$b^{[l]}$ of shape $(n_l, 1)$ (it broadcasts over the $m$ columns). Note that `torch.nn` uses the transposed
convention `(batch, features)` and computes `X @ W.T + b`; we keep the lecture's convention here.

**Forward pass** of one layer:

$$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}, \qquad A^{[l]} = g\left(Z^{[l]}\right).$$

**Cost** of a batch is the mean of the per-sample losses:

$$J = \frac{1}{m} \sum_{i=1}^{m} L\left(\hat y_i, y_i\right).$$

**Backward pass.** Backpropagation is the chain rule applied layer by layer, from the loss down to the first layer.
Every module receives the gradient with respect to its *output* and returns the gradient with respect to its
*input*; layers with parameters additionally store the gradient with respect to their parameters.

- Through an activation, $A = g(Z)$ is elementwise, so
  $$dZ = dA \odot g'(Z).$$
- Through a linear layer, with the per-sample gradient $dZ$ of shape $(n_l, m)$ (column $i$ is
  $\partial L_i / \partial Z_{:,i}$), the gradients of the **cost** $J$ with respect to the parameters and the
  per-sample gradient handed to the previous layer are
  $$dW = \frac{\partial J}{\partial W^{[l]}} = \frac{1}{m}\, dZ\, {A^{[l-1]}}^{T}, \qquad
    db = \frac{\partial J}{\partial b^{[l]}} = \frac{1}{m} \sum_{i=1}^{m} dZ_{:, i}, \qquad
    dA^{[l-1]} = {W^{[l]}}^{T} dZ.$$

**Reduction convention (keep it exactly, the verification cells depend on it).** The gradient that flows *between*
layers is the **per-sample** gradient: column $i$ of `dZ` is $\partial L_i / \partial Z_{:,i}$, without any $1/m$.
The averaging over the batch happens only where **parameter** gradients are formed, in `Linear.backward`, because the
cost $J$ is the mean of the losses. Consequently the loss module's `backward(input, target)` returns
$\partial L_i / \partial \hat y_i$ with the same shape as $\hat y$, and `Linear.backward` applies the $1/m$ to `dW`
and `db` but **not** to `dA_prev` (that one is still per-sample).

**Derivatives of the activations you will need:**

$$\sigma'(z) = \sigma(z)\,(1 - \sigma(z)), \qquad
  \tanh'(z) = 1 - \tanh^2(z), \qquad
  \mathrm{ReLU}'(z) = \mathbb{1}[z > 0], \qquad
  \mathrm{LeakyReLU}'_\alpha(z) = \begin{cases} 1 & z > 0 \\ \alpha & z \le 0 \end{cases}$$

## 2. The `Module` base class (given)

Every building block of our framework (layers, activations, losses, the whole model) is a `Module`: it has a
`forward` and a `backward` method and may contain named sub-modules. This is a stripped-down cousin of
`torch.nn.Module`. Two differences: our `add_module(module, name)` takes the module first (PyTorch
takes the name first), and there is no autograd. Whatever a module needs in `backward`, it must remember in `forward`.

In [ ]:
class Module:
    """Minimal base class of our little framework: forward, backward and named sub-modules."""

    def __init__(self) -> None:
        self.modules: OrderedDict[str, "Module"] = OrderedDict()

    def add_module(self, module: "Module", name: str) -> None:
        if not name or "." in name:
            raise KeyError(f"invalid module name {name!r}")
        if name in self.modules:
            raise KeyError(f"module {name!r} already exists")
        self.modules[name] = module

    def forward(self, *args, **kwargs):
        raise NotImplementedError

    def backward(self, *args, **kwargs):
        raise NotImplementedError

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def __repr__(self) -> str:
        return f"{type(self).__name__}()"

## 3. Linear layer

`forward` is the one you know from last week. It additionally remembers its input (`self.fw_inputs`, this is
$A^{[l-1]}$) and the batch size `self.m`, because both are needed in `backward`.

**3.1 Implement `Linear.backward(dZ)`.** It receives $dZ = \partial L/\partial Z$ of shape `(out_features, m)`
and must

- store `self.dW` $= \frac{1}{m}\, dZ\, A_{prev}^{T}$, shape `(out_features, in_features)`,
- store `self.db` $= \frac{1}{m} \sum_{\text{columns}} dZ$, shape `(out_features, 1)` (use `keepdim=True`),
- return $dA_{prev} = W^{T} dZ$, shape `(in_features, m)`, **without** the $1/m$.

Hints: `@` is matrix multiplication, `.T` transposes, `tensor.sum(dim=1, keepdim=True)` sums over the samples.
Check the shapes with the formulas: `(out, m) @ (m, in) -> (out, in)`.

In [ ]:
class Linear(Module):
    """Fully connected layer, column convention: Z = W @ A_prev + b with A_prev of shape (in_features, m)."""

    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = torch.randn(out_features, in_features, dtype=DTYPE)
        self.b = torch.zeros(out_features, 1, dtype=DTYPE)   # a column: it broadcasts over the m samples
        self.dW = torch.zeros_like(self.W)                    # gradient buffers, filled by backward()
        self.db = torch.zeros_like(self.b)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_inputs = input        # A_prev: needed for dW in backward
        self.m = input.shape[1]       # number of samples in the batch
        return self.W @ input + self.b

    def backward(self, dZ: torch.Tensor) -> torch.Tensor:
        # TODO: given dZ = dL/dZ of shape (out_features, m):
        #   - store self.dW = (1/m) * dZ @ A_prev^T             -> shape (out_features, in_features)
        #   - store self.db = (1/m) * sum of dZ over the samples -> shape (out_features, 1), keepdim=True
        #   - return dA_prev = W^T @ dZ                          -> shape (in_features, m), no 1/m here
        self.dW = ...
        self.db = ...
        return ...

    def __repr__(self) -> str:
        return f"Linear(in_features={self.in_features}, out_features={self.out_features})"

**3.2 Verification.** We feed a random upstream gradient `dZ` into your `backward` and compare with autograd.

The trick: for the scalar $S = \sum (Z \odot dZ)$ with `dZ` treated as a constant, $\partial S / \partial Z = dZ$
exactly, so autograd propagates precisely the upstream gradient we handed to your `backward`. Because $S$ is a sum
and not a mean, autograd's `W_.grad` equals $m \cdot dW$ in our convention, hence the division by `m` below;
`X_.grad` needs no correction because `dA_prev` is per-sample.

In [ ]:
torch.manual_seed(0)
m = 7
layer = Linear(3, 4)
X = torch.randn(3, m, dtype=DTYPE)
dZ = torch.randn(4, m, dtype=DTYPE)          # a made-up upstream gradient dL/dZ

Z = layer(X)                                 # forward first: backward relies on the stored input
dA_prev = layer.backward(dZ)

# Autograd reference: the same computation on leaf copies that record gradients.
W_ = layer.W.clone().requires_grad_(True)
b_ = layer.b.clone().requires_grad_(True)
X_ = X.clone().requires_grad_(True)
Z_ = W_ @ X_ + b_
(Z_ * dZ).sum().backward()

results = [
    check("Linear dW", layer.dW, W_.grad / m),
    check("Linear db", layer.db, b_.grad / m),
    check("Linear dA_prev", dA_prev, X_.grad),
]
assert all(results), "Linear.backward does not match autograd: fix the TODO in 3.1"

## 4. Activation functions

Activations have no parameters, so their `backward(dA)` only has to return $dZ = dA \odot g'(Z)$. The `forward`
methods are given and store their input $Z$ in `self.fw_input`.

**4.1 Implement `backward` for `Sigmoid`, `Tanh`, `ReLU` and `LeakyReLU`** using the derivatives from the recap.

Hints:
- For Sigmoid and Tanh the derivative is easiest to write in terms of the *output* $a = g(z)$:
  recompute it with `a = self.forward(self.fw_input)`.
- For ReLU the derivative is a mask: `(self.fw_input > 0)` is a boolean tensor, `.to(dA.dtype)` turns it into 0/1.
- For LeakyReLU use `torch.where(condition, tensor_if_true, tensor_if_false)`; `torch.full_like(z, value)` creates a
  tensor filled with a constant.
- All four are one-liners (plus the recomputation of `a`). No loops.

In [ ]:
class Sigmoid(Module):
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_input = input
        return 1.0 / (1.0 + torch.exp(-input))

    def backward(self, dA: torch.Tensor) -> torch.Tensor:
        # TODO: return dA * sigma'(Z) where sigma'(z) = sigma(z) * (1 - sigma(z)) and Z = self.fw_input
        a = ...
        return ...


class Tanh(Module):
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_input = input
        return torch.tanh(input)

    def backward(self, dA: torch.Tensor) -> torch.Tensor:
        # TODO: return dA * tanh'(Z) where tanh'(z) = 1 - tanh(z)^2
        a = ...
        return ...


class ReLU(Module):
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_input = input
        return torch.clamp(input, min=0.0)

    def backward(self, dA: torch.Tensor) -> torch.Tensor:
        # TODO: return dA * 1[Z > 0]   (the gradient passes only where the input was positive)
        return ...


class LeakyReLU(Module):
    def __init__(self, negative_slope: float = 0.01) -> None:
        super().__init__()
        self.negative_slope = negative_slope

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_input = input
        return torch.where(input > 0, input, self.negative_slope * input)

    def backward(self, dA: torch.Tensor) -> torch.Tensor:
        # TODO: return dA * g'(Z) with g'(z) = 1 for z > 0 and negative_slope otherwise
        return ...

    def __repr__(self) -> str:
        return f"LeakyReLU(negative_slope={self.negative_slope})"

**4.2 Verification.** For each activation we compare `forward` with the PyTorch function and `backward` with the
autograd gradient of $\sum g(Z) \odot dA$ (the same trick as in 3.2: this scalar has exactly `dA` as its
gradient with respect to $g(Z)$).

In [ ]:
torch.manual_seed(0)
Z = torch.randn(4, 6, dtype=DTYPE)
dA = torch.randn(4, 6, dtype=DTYPE)

references = {
    "Sigmoid": (Sigmoid(), torch.sigmoid),
    "Tanh": (Tanh(), torch.tanh),
    "ReLU": (ReLU(), torch.relu),
    "LeakyReLU": (LeakyReLU(0.01), lambda z: F.leaky_relu(z, negative_slope=0.01)),
}
results = []
for name, (mine, ref_fn) in references.items():
    A = mine(Z)                    # forward first: backward relies on the stored input
    dZ = mine.backward(dA)
    Z_ = Z.clone().requires_grad_(True)
    (ref_fn(Z_) * dA).sum().backward()
    results.append(check(f"{name} forward", A, ref_fn(Z)))
    results.append(check(f"{name} backward", dZ, Z_.grad))
assert all(results), "some activation backward passes do not match autograd: fix the TODOs in 4.1"

## 5. Loss functions

**Loss vs. cost.** The *loss* $L_i = L(\hat y_i, y_i)$ is defined per sample; the *cost* $J = \frac{1}{m}\sum_i L_i$
is one number for the whole batch and is what gradient descent minimises. In our framework the loss module's
`forward` returns the **per-sample** losses (a tensor with the shape of $\hat y$; take `.mean()` yourself to get
$J$) and `backward` returns the per-sample derivative $\partial L_i / \partial \hat y_i$ with the same shape.
The $1/m$ is applied later, inside `Linear.backward` (see the reduction convention in the recap).

**Squared error** (regression, or a sigmoid output treated as a real number):

$$L = (y - \hat y)^2, \qquad \frac{\partial L}{\partial \hat y} = -2\,(y - \hat y).$$

**Binary cross-entropy** (binary classification, $\hat y \in (0, 1)$ is the predicted probability of class 1):

$$L = -\bigl(y \log \hat y + (1 - y)\log(1 - \hat y)\bigr), \qquad
  \frac{\partial L}{\partial \hat y} = -\left(\frac{y}{\hat y} - \frac{1 - y}{1 - \hat y}\right).$$

**Why clamp, and what the clamp does not fix.** A sigmoid can return exactly `0.0` or `1.0` in floating point. Then
$\log 0 = -\infty$ in the forward pass and the division by $\hat y$ or $1 - \hat y$ in the backward pass produces
`inf`/`nan`, which then poisons every parameter of the network. Clamping $\hat y$ to $[\varepsilon, 1 - \varepsilon]$
before taking logs and dividing keeps the loss and its gradient **finite** (`torch.nn.BCELoss` does something similar
by clamping $\log \hat y$ to $\ge -100$). It does **not** make the gradient correct. For a saturated sigmoid the clamped
$\partial L / \partial \hat y$ is huge (about $\pm 1 / \varepsilon$) while $\sigma'(z) = \hat y\,(1 - \hat y)$ is
tiny, and their product, the gradient with respect to the logit $z$, comes out near zero (exactly zero once $\hat y$
underflows to `0.0`). The true value is $\partial L / \partial z = \hat y - y$, i.e. $\pm 1$: the network gets almost
no learning signal precisely where it is most wrong. The fix used in practice is to fuse the sigmoid and the
cross-entropy into one module that takes the logit $z$ and never forms $\hat y$ explicitly
(`torch.nn.BCEWithLogitsLoss`, computed with the log-sum-exp trick); its gradient is simply $\hat y - y$. You will
use it from week 4 on. Here we keep `Sigmoid` and `BCELoss` separate, because this notebook is about verifying
every module on its own.

**5.1 Implement `SELoss` (forward and backward).**
**5.2 Implement `BCELoss` (forward and backward).** Clamp with `torch.clamp(input, EPS, 1 - EPS)` in both methods.

Hints: everything is elementwise (`*`, `/`, `torch.log`), no sums or means inside the loss module.

In [ ]:
class SELoss(Module):
    """Squared error per sample: L_i = (y_i - yhat_i)^2 (no averaging here, see the text above)."""

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # TODO: return the elementwise squared error (target - input)^2, same shape as input
        return ...

    def backward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # TODO: return dL/d input = -2 * (target - input), same shape as input
        return ...


class BCELoss(Module):
    """Binary cross-entropy per sample: L_i = -(y_i log yhat_i + (1 - y_i) log(1 - yhat_i))."""

    EPS = 1e-12   # predictions are clamped to [EPS, 1 - EPS] so that log(0) and x/0 never happen

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # TODO: clamp input to [EPS, 1 - EPS], then return the elementwise BCE (same shape as input)
        y_hat = ...
        return ...

    def backward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # TODO: clamp as in forward, then return dL/d y_hat = -(y / y_hat - (1 - y) / (1 - y_hat))
        y_hat = ...
        return ...

**5.3 Verification.** `reduction="none"` makes the PyTorch losses return per-sample values like ours. For the
gradient we differentiate the **sum** of the PyTorch loss, so that autograd returns $\partial L_i/\partial \hat y_i$
without a $1/m$, exactly what our `backward` promises. The last check only confirms that the clamp keeps the loss
and the gradient **finite** at $\hat y = 0$ and $\hat y = 1$; it says nothing about the gradient being *correct*
there (it is not, see above).

In [ ]:
torch.manual_seed(0)
Y = torch.randint(0, 2, (1, 8)).to(DTYPE)                  # binary targets 0/1
Y_hat = torch.rand(1, 8, dtype=DTYPE) * 0.9 + 0.05         # predictions inside (0.05, 0.95)

se = SELoss()
Yh_ = Y_hat.clone().requires_grad_(True)
F.mse_loss(Yh_, Y, reduction="sum").backward()
results = [
    check("SELoss forward", se(Y_hat, Y), F.mse_loss(Y_hat, Y, reduction="none")),
    check("SELoss backward", se.backward(Y_hat, Y), Yh_.grad),
]

bce = BCELoss()
Yh_ = Y_hat.clone().requires_grad_(True)
F.binary_cross_entropy(Yh_, Y, reduction="sum").backward()
results += [
    check("BCELoss forward", bce(Y_hat, Y), F.binary_cross_entropy(Y_hat, Y, reduction="none")),
    check("BCELoss backward", bce.backward(Y_hat, Y), Yh_.grad),
]

# The clamp at work: finite (not correct, see the text above) values for the worst possible predictions
# (0 for a positive, 1 for a negative).
extreme_pred = torch.tensor([[0.0, 1.0]], dtype=DTYPE)
extreme_true = torch.tensor([[1.0, 0.0]], dtype=DTYPE)
extreme_loss = bce(extreme_pred, extreme_true)
extreme_grad = bce.backward(extreme_pred, extreme_true)
loss_is_finite = torch.is_tensor(extreme_loss) and bool(torch.isfinite(extreme_loss).all())
grad_is_finite = torch.is_tensor(extreme_grad) and bool(torch.isfinite(extreme_grad).all())
finite = loss_is_finite and grad_is_finite
print(f"{'OK        ' if finite else 'MISMATCH  '}BCELoss is finite at y_hat = 0 and 1 (the clamp keeps it finite, not correct)")
assert all(results), "SELoss / BCELoss do not match torch.nn.functional: fix the TODOs in 5.1 / 5.2"
assert finite, "BCELoss returns inf/nan at y_hat = 0 or 1: clamp the prediction in forward AND backward"

## 6. The `Model` container

`Model` keeps its layers in the ordered dictionary `self.modules` (filled with `add_module`). `forward` is given: it
passes the data through the modules in insertion order.

**6.1 Implement `Model.backward(dZ)`.** It receives $\partial L/\partial \hat Y$ from the loss and must visit the
modules in **reverse** order, calling each module's `backward` with the gradient returned by the module after it.
Return the final gradient (with respect to the network input); it is rarely needed for training but useful for
checking.

Hint: `reversed(self.modules.items())` iterates the `(name, module)` pairs from the last to the first.

In [ ]:
class Model(Module):
    """A sequential container: forward runs the modules in insertion order, backward in reverse order."""

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        for name, module in self.modules.items():
            input = module(input)
        return input

    def backward(self, dZ: torch.Tensor) -> torch.Tensor:
        # TODO: propagate dZ through the modules in REVERSE order (last module first);
        #       every module.backward(...) returns the gradient for the module before it.
        #       Return the gradient with respect to the network input.
        for name, module in reversed(self.modules.items()):
            dZ = ...
        return dZ

    def __repr__(self) -> str:
        body = "\n".join(f"  ({name}): {module}" for name, module in self.modules.items())
        return f"Model(\n{body}\n)"

**6.2 Verification.** The helper `torch_reference` (given) rebuilds the forward pass of any of our models with plain
PyTorch operations on leaf copies that record gradients. We use it here on a tiny network, and again at the end of
the notebook. As in 3.2 the reference scalar is a sum, so parameter gradients are divided by `m`.

In [ ]:
def torch_reference(model: Model, X: torch.Tensor):
    """Rebuild the forward pass of `model` with plain torch ops on leaf copies that record gradients.

    Returns (output, X_leaf, params) where params[name] = (W_leaf, b_leaf) for every Linear layer.
    After calling .backward() on a scalar built from `output`, the .grad fields hold autograd's answer.
    """
    X_leaf = X.clone().requires_grad_(True)
    A = X_leaf
    params: dict[str, tuple[torch.Tensor, torch.Tensor]] = {}
    for name, module in model.modules.items():
        if isinstance(module, Linear):
            W = module.W.clone().requires_grad_(True)
            b = module.b.clone().requires_grad_(True)
            params[name] = (W, b)
            A = W @ A + b
        elif isinstance(module, Sigmoid):
            A = torch.sigmoid(A)
        elif isinstance(module, Tanh):
            A = torch.tanh(A)
        elif isinstance(module, ReLU):
            A = torch.relu(A)
        elif isinstance(module, LeakyReLU):
            A = F.leaky_relu(A, negative_slope=module.negative_slope)
        else:
            raise TypeError(f"no autograd reference for {type(module).__name__}")
    return A, X_leaf, params


torch.manual_seed(0)
tiny = Model()
tiny.add_module(Linear(2, 3), "Dense_1")
tiny.add_module(ReLU(), "ReLU_1")
tiny.add_module(Linear(3, 1), "Dense_2")
tiny.add_module(Sigmoid(), "Sigmoid")
print(tiny)

m = 5
X = torch.randn(2, m, dtype=DTYPE)
dY = torch.randn(1, m, dtype=DTYPE)          # a made-up upstream gradient dL/dY_hat
Y_hat = tiny(X)
dX = tiny.backward(dY)

out, X_leaf, params = torch_reference(tiny, X)
(out * dY).sum().backward()
results = [check("Model dX", dX, X_leaf.grad)]
for name, (W_, b_) in params.items():
    results.append(check(f"{name}.dW", tiny.modules[name].dW, W_.grad / m))
    results.append(check(f"{name}.db", tiny.modules[name].db, b_.grad / m))
assert all(results), "Model.backward does not match autograd: fix the TODO in 6.1 (reverse order!)"

## 7. The Circles dataset (given)

Points are drawn uniformly from the square $[-1, 1]^2$; a point belongs to class 1 when its (noisy) distance from
the origin is larger than `radius`. The data comes in our column convention: `X` has shape `(2, m)` and `Y` has
shape `(1, m)`.

In [ ]:
def dataset_circles(m: int = 128, radius: float = 0.7, noise: float = 0.0) -> tuple[torch.Tensor, torch.Tensor]:
    """m points uniform in [-1, 1]^2; label 1 if the (noisy) distance from the origin exceeds `radius`.

    Returns X of shape (2, m) and Y of shape (1, m), both float64 (column convention).
    """
    X = torch.rand(2, m, dtype=DTYPE) * 2.0 - 1.0
    N = (torch.rand(2, m, dtype=DTYPE) - 0.5) * noise
    X_noisy = X + N
    R = torch.sqrt((X_noisy ** 2).sum(dim=0, keepdim=True))
    Y = (R > radius).to(DTYPE)
    return X, Y


def draw_dataset(X: torch.Tensor, Y: torch.Tensor, title: str = "Circles dataset") -> None:
    plt.figure(figsize=(5, 5))
    plt.scatter(X[0], X[1], c=Y[0], cmap="RdBu", edgecolors="k", s=25)
    plt.gca().set_aspect("equal")
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title(title)
    plt.show()


torch.manual_seed(SEED)
X, Y = dataset_circles(m=128, radius=0.7, noise=0.0)
print("X:", tuple(X.shape), " Y:", tuple(Y.shape), " class 1 fraction:", Y.mean().item())
draw_dataset(X, Y)

## 8. Putting it together: forward, loss, backward, gradient check

We build the same MLP as in the original lab (2-3-4-5-1 with Tanh hidden activations and a Sigmoid output) and run
one forward pass on the whole dataset. Nothing is trained here; that is Task 3.

In [ ]:
torch.manual_seed(SEED)
mlp = Model()
mlp.add_module(Linear(2, 3), "Dense_1")
mlp.add_module(Tanh(), "Tanh_1")
mlp.add_module(Linear(3, 4), "Dense_2")
mlp.add_module(Tanh(), "Tanh_2")
mlp.add_module(Linear(4, 5), "Dense_3")
mlp.add_module(Tanh(), "Tanh_3")
mlp.add_module(Linear(5, 1), "Dense_4_out")
mlp.add_module(Sigmoid(), "Sigmoid")
print(mlp)

Y_hat = mlp(X)
print("X:", tuple(X.shape), " Y:", tuple(Y.shape), " Y_hat:", tuple(Y_hat.shape))

**8.1 Choose the loss and run the backward pass.**

1. The output layer is a sigmoid and the labels are 0/1: pick the matching loss from section 5 and instantiate it as
   `loss_fn`. `loss` is then the per-sample loss tensor of shape `(1, m)` and `cost` its mean.
2. Run the backward pass: the loss module turns `(Y_hat, Y)` into $\partial L / \partial \hat Y$, and the model
   propagates it down through all layers. After this call every `Linear` layer holds its `dW` and `db`.

In [ ]:
# TODO (1): instantiate the loss that matches a sigmoid output with 0/1 labels
loss_fn = ...
assert isinstance(loss_fn, Module), "TODO (1): loss_fn must be an instance of one of the loss modules from section 5"
loss = loss_fn(Y_hat, Y)          # per-sample losses, shape (1, m)
cost = loss.mean()                # one number for the whole batch
print(f"loss shape: {tuple(loss.shape)}   cost J = {cost.item():.4f}")

# TODO (2): backward pass: feed the loss gradient dL/dY_hat into the model
dY_hat = ...
dX = ...
print("dW of Dense_1:\n", mlp.modules["Dense_1"].dW)
print("db of Dense_4_out:", mlp.modules["Dense_4_out"].db.squeeze().item())

assert isinstance(loss_fn, BCELoss), "a sigmoid output with 0/1 labels calls for the binary cross-entropy"
assert tuple(loss.shape) == (1, X.shape[1]), f"loss should be per-sample with shape (1, m), got {tuple(loss.shape)}"
assert torch.is_tensor(dX) and dX.shape == X.shape, "mlp.backward(...) must return the gradient with respect to X, shape (2, m)"

**8.2 Gradient check (given).** `gradient_check` perturbs every single weight and bias by $\pm\varepsilon$, recomputes
the cost $J = \text{mean}(\text{loss})$ and forms the central difference

$$\frac{\partial J}{\partial \theta_k} \approx \frac{J(\theta_k + \varepsilon) - J(\theta_k - \varepsilon)}{2\varepsilon}.$$

It then compares the vector of all numerical derivatives with the analytic `dW`, `db` from your backward pass
using the relative difference $\|g_{\text{analytic}} - g_{\text{numeric}}\| \,/\, (\|g_{\text{analytic}}\| + \|g_{\text{numeric}}\|)$.
With `float64` a correct implementation lands far below the threshold $2 \cdot 10^{-7}$ (typically around $10^{-9}$).
Call it right after `mlp.backward(...)`, because it reads the stored gradients.

In [ ]:
def gradient_check(network: Model, loss_function: Module, X: torch.Tensor, Y: torch.Tensor,
                   epsilon: float = 1e-7, threshold: float = 2e-7) -> float:
    """Compare the analytic dW, db stored in the layers with central finite differences of J = mean(loss).

    Costs two forward passes per parameter, so use it on small networks only. Returns the relative difference.
    """
    analytic, numeric = [], []
    for name, layer in network.modules.items():
        if not (hasattr(layer, "W") and hasattr(layer, "dW")):
            continue
        for param, grad in ((layer.W, layer.dW), (layer.b, layer.db)):
            flat = param.view(-1)              # a view: writing into it changes the layer's parameter in place
            for k in range(flat.numel()):
                original = flat[k].item()
                flat[k] = original + epsilon
                J_plus = loss_function(network(X), Y).mean()
                flat[k] = original - epsilon
                J_minus = loss_function(network(X), Y).mean()
                flat[k] = original
                central_difference = (J_plus - J_minus) / (2.0 * epsilon)
                numeric.append(central_difference.item())
                analytic.append(grad.reshape(-1)[k].item())
    network(X)                                 # leave the cached activations consistent with the true parameters

    analytic_t = torch.tensor(analytic, dtype=DTYPE)
    numeric_t = torch.tensor(numeric, dtype=DTYPE)
    norm_of_difference = torch.linalg.norm(analytic_t - numeric_t)
    norm_scale = torch.linalg.norm(analytic_t) + torch.linalg.norm(numeric_t)
    difference = (norm_of_difference / norm_scale).item()
    if not math.isfinite(difference) or difference > threshold:
        print(f"\033[91mThere is a mistake in the backward propagation! "
              f"difference = {difference:.3e} > threshold = {threshold:.0e}\033[0m")
    else:
        print(f"\033[92mYour backward propagation works! "
              f"difference = {difference:.3e} <= threshold = {threshold:.0e}\033[0m")
    return difference


difference = gradient_check(mlp, loss_fn, X, Y)
assert math.isfinite(difference) and difference <= 2e-7, \
    f"gradient check failed (difference = {difference:.3e}): the autograd cross-check below tells you which layer is wrong"

**8.3 Bonus: autograd cross-check of every layer (given).** The numerical check gives one number for the whole
network. If it fails, this cell tells you *which* layer is wrong: it differentiates PyTorch's own
`binary_cross_entropy(..., reduction="mean")` through `torch_reference` and compares layer by layer. The mean
reduction already contains the $1/m$, so no extra division this time.

In [ ]:
out, _, params = torch_reference(mlp, X)
F.binary_cross_entropy(out, Y, reduction="mean").backward()
results = []
for name, (W_, b_) in params.items():
    results.append(check(f"{name}.dW", mlp.modules[name].dW, W_.grad))
    results.append(check(f"{name}.db", mlp.modules[name].db, b_.grad))
assert all(results), "some layer gradients do not match autograd (see the MISMATCH lines above)"

## 9. Check your understanding

Answer briefly in the markdown cells (double-click to edit).

**9.1 Why does this notebook use `torch.float64`? What would happen to the gradient check with `float32`?**

_Your answer here._

**9.2 The derivative of ReLU is not defined at $z = 0$. What does your implementation return there, what does
PyTorch do, and can this break the gradient check?**

_Your answer here._

**9.3 The gradient check prints a difference of, say, $3 \cdot 10^{-3}$. What kinds of mistakes typically produce a
value like this, and how would you locate the faulty layer?**

_Your answer here._

**9.4 `Linear.backward` divides by `m`, but the `backward` methods of the activations and of the loss do not.
Explain why this is consistent.**

_Your answer here._